# **G2P Evaluation on the Aquino et al. Speech Corpus**

This notebook trains and evaluates WFST grapheme-to-phoneme (G2P) models with Phonetisaurus on
the Tagalog speech-corpus transcriptions of Aquino et al. (2019).

These are the project's first experiments. They set the WFST baseline, the default Phonetisaurus
settings with `--seq2_del`, and found that a grid search over the settings doesn't improve on it.
`Wik_eval.ipynb` uses that baseline on the Wiktionary data, and `G2P_compare.ipynb` makes the
final choice of G2P.

Each recording in the corpus has two files: a `.log` file with the utterance as written, and a
`.csv` file with its phonetic transcription. A G2P model learns how spelling maps to sound from
the training split, then predicts the transcriptions of the test split. The predictions are scored
by phone error rate (PER) against the manually verified transcriptions in `testref`.

The corpus comes in two versions, `data/_small` and `data/_clean`, a cleaned copy of the same
recordings. Run the notebook inside the Docker container (`./run_phonetisaurus.ps1`), where
Phonetisaurus is installed.

Trained models are saved to `train/`, predictions to `output/` and error reports to `errors/`.
Scratch files (file lists, word lists, training by-products) live in a temporary directory that
the last cell deletes.

**The process at a glance**

1. **List** the corpus files of each split.
2. **Build** the training dictionaries and the test word list.
3. **Train and score** three models on the clean test split: a baseline, a model trained on
   cleaned data, and a tuned model.
4. **Analyze** the errors and draw conclusions.
5. **Extract** the phoneme inventory of the clean model.
6. **Search** the hyperparameters with cross-validation, then retrain and test the best
   configuration.
7. **Clean up** the temporary files.

In [ ]:
import subprocess
import os
import sys
import itertools
import tempfile
import random
import pandas as pd
from pathlib import Path

sys.dont_write_bytecode = True  # no __pycache__ in ../utils
sys.path.append(os.path.abspath("../utils"))
from preprocessing_tools import *
from calculate_g2p_accuracy import *
from alignment import align_trxn
from train_tools import train_model

# Scratch files (file lists, word lists, training by-products, hypothesis CSVs)
# live here; the directory is deleted by the Cleanup cell at the end (or when
# the kernel exits).
tmp = tempfile.TemporaryDirectory(prefix="aquino_eval_")
tmp_dir = Path(tmp.name)

## 1. List the data files

The `scripts/` folder has one `get-*.sh` script per file list. For example,
`get-clean-log-train-files.sh` lists the `.log` files of the clean training split. Each script
prints its list, and the notebook saves it to the temporary directory
(`paths/clean-log-train-files.txt`).

The scripts are first converted to Linux line endings. A Windows checkout gives them `\r\n` line
endings, which `bash` can't run.

In [2]:
current_dir = Path().resolve()

scripts_dir = current_dir.parent / "scripts"
sh_files = [f for f in os.listdir(scripts_dir) if f.endswith(".sh")]

for file in sh_files:

    file_dir = scripts_dir / file

    try:
        # Read the file in binary mode
        with open(file_dir, "rb") as f:
            content = f.read()
        
        # Replace Windows line endings (\r\n) with Linux line endings (\n)
        content = content.replace(b"\r\n", b"\n")
        
        # Write the corrected content back
        with open(file_dir, "wb") as f:
            f.write(content)
            
        print("Success! Converted line endings to LF.")
    except FileNotFoundError:
        print(f"Could not find the file at {file_dir}")

Success! Converted line endings to LF.
Success! Converted line endings to LF.
Success! Converted line endings to LF.
Success! Converted line endings to LF.
Success! Converted line endings to LF.
Success! Converted line endings to LF.
Success! Converted line endings to LF.
Success! Converted line endings to LF.
Success! Converted line endings to LF.
Success! Converted line endings to LF.
Success! Converted line endings to LF.
Success! Converted line endings to LF.


In [3]:
paths_dir = tmp_dir / "paths"
paths_dir.mkdir()

getsh_files = sorted(f for f in os.listdir(scripts_dir) if f.endswith(".sh") and f.startswith("get-"))

for file in getsh_files:
    try:
        # Each script prints a file list; save it as e.g. paths/clean-csv-train-files.txt
        result = subprocess.run(
            ["bash", file],
            cwd=scripts_dir,
            check=True,
            capture_output=True,
            text=True
        )
        list_file = paths_dir / (file.removeprefix("get-").removesuffix(".sh") + ".txt")
        list_file.write_text(result.stdout)
        print(f"Listed {file} into {list_file.name}")

    except subprocess.CalledProcessError as e:
        print(f"Script failed with exit code {e.returncode}")
        print(f"Error details:\n{e.stderr}")
    except FileNotFoundError:
        print(f"Could not find the scripts directory at: {scripts_dir}")

You should now see get-clean-csv-test-files.sh inside the /workspaces/Taglog-G2P/notebook/paths folder.
You should now see get-clean-csv-testref-files.sh inside the /workspaces/Taglog-G2P/notebook/paths folder.
You should now see get-clean-csv-train-files.sh inside the /workspaces/Taglog-G2P/notebook/paths folder.
You should now see get-clean-csv-valref-files.sh inside the /workspaces/Taglog-G2P/notebook/paths folder.
You should now see get-clean-log-test-files.sh inside the /workspaces/Taglog-G2P/notebook/paths folder.
You should now see get-clean-log-train-files.sh inside the /workspaces/Taglog-G2P/notebook/paths folder.
You should now see get-clean-log-valref-files.sh inside the /workspaces/Taglog-G2P/notebook/paths folder.
You should now see get-csv-test-files.sh inside the /workspaces/Taglog-G2P/notebook/paths folder.
You should now see get-csv-testref-files.sh inside the /workspaces/Taglog-G2P/notebook/paths folder.
You should now see get-csv-train-files.sh inside the /workspaces

The lists are then grouped by split. Each training split is a pair, `[csv list, log list]`. The
reference splits only need their `.csv` list, and the test splits only their `.log` list, since
the test transcriptions are what the models predict.

In [4]:
# Pairs are [csv list, log list]. The test sets only need their log list: their
# CSVs are the G2P hypotheses, which create_trxn writes later.
train_files   = [paths_dir / "csv-train-files.txt", paths_dir / "log-train-files.txt"]
testref_files = [paths_dir / "csv-testref-files.txt"]
test_log      = paths_dir / "log-test-files.txt"

# Cleaned
clean_train_files   = [paths_dir / "clean-csv-train-files.txt", paths_dir / "clean-log-train-files.txt"]
clean_testref_files = [paths_dir / "clean-csv-testref-files.txt"]
clean_test_log      = paths_dir / "clean-log-test-files.txt"


## 2. Build the dictionaries and word lists

`create_dict()` pairs each word of an utterance (from the `.log` file) with its transcription
(from the `.csv` file) and writes the unique pairs as a pronunciation dictionary. Utterances whose
word count doesn't match their transcription are printed and skipped.

| File | Built from | Used for |
| --- | --- | --- |
| `dicts/train.dict` | `data/_small` training split | the tuned model and the grid search |
| `dicts/clean_train.dict` | `data/_clean` training split | not used further in this notebook |
| `clean_test.wlist` (temporary) | `data/_clean` test split | the words every model is tested on |
| `wik_test.wlist` (temporary) | `dicts/test.tsv` | not used further in this notebook |

In [5]:
dicts_dir = current_dir / "dicts"

create_dict(train_files[1], train_files[0], dicts_dir / "train.dict")

../data/_small/train/TGL/0812.110816.021250.csv
../data/_small/train/TGL/2359.110818.085852.csv


In [6]:
dicts_dir = current_dir / "dicts"

create_dict(clean_train_files[1], clean_train_files[0], dicts_dir / "clean_train.dict")

../data/_clean/train/TGL/0812.110816.021250.csv
../data/_clean/train/TGL/2359.110818.085852.csv


In [ ]:
wlist_dir = current_dir / "wlist"

create_wlist(clean_test_log, tmp_dir / "clean_test.wlist")


In [17]:
df = pd.read_csv(dicts_dir / "test.tsv", sep="\t")

with open(dicts_dir / "test.tsv", "r") as fin, open(tmp_dir / "wik_test.wlist", "w") as fout:
    for line in fin:
        first_col = line.rstrip("\n").split("\t")[0]
        fout.write(first_col + "\n")

## 3. Baseline model

The baseline follows Aquino et al. (2019), specifically their monolingual Tagalog G2P. It uses the
default Phonetisaurus settings with only `--seq2_del` turned on, which lets a grapheme map to no
phoneme (so a digraph like *ng* can become a single /ŋ/). It is trained on `dicts/train.tsv`.

`train_model()` keeps only the finished model, `train/model.fst`. The alignment corpus and
language model that Phonetisaurus also writes stay in the temporary directory.

In [ ]:
model_dir = current_dir / "train"

result = train_model(dicts_dir / "train.tsv", "model", model_dir, tmp_dir / "train", "--seq2_del")

print(result.stderr)

INFO:phonetisaurus-train:2026-06-10 02:50:55:  Checking command configuration...
INFO:phonetisaurus-train:2026-06-10 02:50:55:  Checking lexicon for reserved characters: '}', '|', '_'...
INFO:phonetisaurus-train:2026-06-10 02:50:55:  Aligning lexicon...
INFO:phonetisaurus-train:2026-06-10 02:50:57:  Training joint ngram model...
INFO:phonetisaurus-train:2026-06-10 02:50:57:  Converting ARPA format joint n-gram model to WFST format...
INFO:phonetisaurus-train:2026-06-10 02:50:58:  G2P training succeeded: train/model.fst



### Score on the test split

The model predicts a transcription for every test word, saved to `output/baseline_output.txt`.
`create_trxn()` then rebuilds each test utterance from those word predictions (a word with no
prediction becomes `#`), and `calculate_accuracy()` aligns it against the reference transcription.

It prints the PER and writes two reports to `errors/`:

- `baseline_errors_errors.txt` lists every utterance with an error, with both transcriptions
  and the edit distance.
- `baseline_errors_ops.txt` counts each gold/predicted phone pair, most frequent first.

In [9]:
model_dir = current_dir / "train"

output_dir = current_dir / "output"
output_dir.mkdir(exist_ok=True)

output_txt = output_dir / "baseline_output.txt"

with open(output_txt, "w") as f:
    subprocess.run(
        [
            "phonetisaurus-apply",
            "--model", str(model_dir / "model.fst"),
            "--word_list", str(tmp_dir / "clean_test.wlist"),
        ],
        stdout=f,
        text=True
    )



In [10]:
hyp_csvs = create_trxn(clean_test_log, output_txt, tmp_dir / "hyp")

calculate_accuracy(hyp_csvs, clean_testref_files[0], "baseline_errors")

../data/_clean/test/TGL/0156.110816.035620.csv
../data/_clean/testref/TGL/0156.110816.035620.csv
total phones in reference: 9254
total edit distance:       460
phone error rate:          0.04970823427706938


({('q', 'q'): 365,
  ('a', 'a'): 2179,
  ('p', 'p'): 331,
  ('o', 'o'): 397,
  ('s', 's'): 385,
  ('t', 't'): 456,
  ('u', 'u'): 323,
  ('h', 'h'): 157,
  ('d', 'd'): 155,
  ('k', 'k'): 341,
  ('i', 'i'): 759,
  ('n', 'n'): 626,
  ('*', 'q'): 19,
  ('l', 'l'): 392,
  ('y', 'y'): 104,
  ('aw', 'aw'): 13,
  ('ng', 'ng'): 626,
  ('ay', 'ay'): 61,
  ('m', 'm'): 394,
  ('r', 'r'): 125,
  ('e', 'e'): 54,
  ('b', 'b'): 207,
  ('g', 'g'): 273,
  ('u', 'o'): 3,
  ('*', 'a'): 108,
  ('ay', 'y'): 10,
  ('er', 'er'): 12,
  ('sh', 'sh'): 8,
  ('ch', 'ch'): 4,
  ('q', '*'): 20,
  ('w', 'w'): 83,
  ('a', '*'): 10,
  ('i', 'ay'): 2,
  ('o', 'u'): 47,
  ('r', 'er'): 3,
  ('g', 'k'): 1,
  ('*', 'g'): 4,
  ('*', 'n'): 100,
  ('s', '*'): 13,
  ('*', 's'): 4,
  ('*', 'i'): 7,
  ('sh', 'y'): 4,
  ('u', '*'): 4,
  ('l', '*'): 4,
  ('o', '*'): 4,
  ('y', '*'): 1,
  ('ng', '*'): 2,
  ('k', '*'): 1,
  ('er', '*'): 1,
  ('b', '*'): 1,
  ('a', 'l'): 1,
  ('ny', 'ny'): 12,
  ('u', 'a'): 5,
  ('i', '*'): 10,
  ('y'

## 4. Model on clean data

Same settings as the baseline, but trained on `dicts/mclean_train.dict`, a training dictionary
cleaned with Claude. The model is saved as `train/clean_model.fst`.

In [11]:
result = train_model(dicts_dir / "mclean_train.dict", "clean_model", model_dir, tmp_dir / "train", "--seq2_del")

print(result.stderr)

INFO:phonetisaurus-train:2026-06-10 02:51:00:  Checking command configuration...
INFO:phonetisaurus-train:2026-06-10 02:51:00:  Checking lexicon for reserved characters: '}', '|', '_'...
INFO:phonetisaurus-train:2026-06-10 02:51:00:  Aligning lexicon...
INFO:phonetisaurus-train:2026-06-10 02:51:01:  Training joint ngram model...
INFO:phonetisaurus-train:2026-06-10 02:51:01:  Converting ARPA format joint n-gram model to WFST format...
INFO:phonetisaurus-train:2026-06-10 02:51:02:  G2P training succeeded: train/clean_model.fst



### Score on the test split

Scored the same way as the baseline. Predictions go to `output/clean_output.txt` and the reports
to `errors/clean_errors_*.txt`.

In [12]:
model_dir = current_dir / "train"

clean_output_txt = current_dir / "output" / "clean_output.txt"

with open(clean_output_txt, "w") as f:
    subprocess.run(
        [
            "phonetisaurus-apply",
            "--model", str(model_dir / "clean_model.fst"),
            "--word_list", str(tmp_dir / "clean_test.wlist"),
        ],
        stdout=f,
        text=True
    )

In [13]:
hyp_csvs = create_trxn(clean_test_log, clean_output_txt, tmp_dir / "hyp")

calculate_accuracy(hyp_csvs, clean_testref_files[0], "clean_errors")

../data/_clean/test/TGL/0156.110816.035620.csv
../data/_clean/testref/TGL/0156.110816.035620.csv
total phones in reference: 9254
total edit distance:       460
phone error rate:          0.04970823427706938


({('q', 'q'): 365,
  ('a', 'a'): 2179,
  ('p', 'p'): 331,
  ('o', 'o'): 397,
  ('s', 's'): 385,
  ('t', 't'): 456,
  ('u', 'u'): 323,
  ('h', 'h'): 157,
  ('d', 'd'): 155,
  ('k', 'k'): 341,
  ('i', 'i'): 759,
  ('n', 'n'): 626,
  ('*', 'q'): 19,
  ('l', 'l'): 392,
  ('y', 'y'): 104,
  ('aw', 'aw'): 13,
  ('ng', 'ng'): 626,
  ('ay', 'ay'): 61,
  ('m', 'm'): 394,
  ('r', 'r'): 125,
  ('e', 'e'): 54,
  ('b', 'b'): 207,
  ('g', 'g'): 273,
  ('u', 'o'): 3,
  ('*', 'a'): 108,
  ('ay', 'y'): 10,
  ('er', 'er'): 12,
  ('sh', 'sh'): 8,
  ('ch', 'ch'): 4,
  ('q', '*'): 20,
  ('w', 'w'): 83,
  ('a', '*'): 10,
  ('i', 'ay'): 2,
  ('o', 'u'): 47,
  ('r', 'er'): 3,
  ('g', 'k'): 1,
  ('*', 'g'): 4,
  ('*', 'n'): 100,
  ('s', '*'): 13,
  ('*', 's'): 4,
  ('*', 'i'): 7,
  ('sh', 'y'): 4,
  ('u', '*'): 4,
  ('l', '*'): 4,
  ('o', '*'): 4,
  ('y', '*'): 1,
  ('ng', '*'): 2,
  ('k', '*'): 1,
  ('er', '*'): 1,
  ('b', '*'): 1,
  ('a', 'l'): 1,
  ('ny', 'ny'): 12,
  ('u', 'a'): 5,
  ('i', '*'): 10,
  ('y'

## 5. Tuned model

This model uses the best configuration from the grid search in section 9. It is trained on
`dicts/train.dict` and tested on the clean test split.

| Setting | Value |
| --- | --- |
| `ngram_order` | 4 |
| `seq1_max` | 1 |
| `seq2_max` | 2 |
| `seq2_del` | on |
| `grow` | on |

The model is saved as `train/tuned_model.fst`.

In [14]:
best_flags = [
    "--ngram_order", str(int(4)),
    "--seq1_max",    str(int(1)),
    "--seq2_max",    str(int(2)),
    "--seq2_del",
    "--grow",
]

print("Training flags:", " ".join(best_flags))

result = train_model(dicts_dir / "train.dict", "tuned_model", model_dir, tmp_dir / "train", *best_flags)
print(result.stderr)

Training command: phonetisaurus-train --lexicon /workspaces/Taglog-G2P/notebook/dicts/train.dict --ngram_order 4 --seq1_max 1 --seq2_max 2 --seq2_del --grow --model /workspaces/Taglog-G2P/notebook/train/tuned_model
INFO:phonetisaurus-train:2026-06-10 02:51:03:  Checking command configuration...
INFO:phonetisaurus-train:2026-06-10 02:51:03:  Checking lexicon for reserved characters: '}', '|', '_'...
INFO:phonetisaurus-train:2026-06-10 02:51:03:  Aligning lexicon...
INFO:phonetisaurus-train:2026-06-10 02:51:05:  Training joint ngram model...
INFO:phonetisaurus-train:2026-06-10 02:51:05:  Converting ARPA format joint n-gram model to WFST format...
INFO:phonetisaurus-train:2026-06-10 02:51:05:  G2P training succeeded: /workspaces/Taglog-G2P/notebook/train/tuned_model.fst



### Score on the test split

Scored the same way as the baseline. Predictions go to `output/tuned_output.txt` and the reports
to `errors/hyperparameters_errors_*.txt`.

In [15]:
tuned_output_txt = current_dir / "output" / "tuned_output.txt"

with open(tuned_output_txt, "w") as f:
    subprocess.run(
        [
            "phonetisaurus-apply",
            "--model",     str(model_dir / "tuned_model.fst"),
            "--word_list", str(tmp_dir / "clean_test.wlist"),
        ],
        stdout=f, text=True
    )

hyp_csvs = create_trxn(clean_test_log, tuned_output_txt, tmp_dir / "hyp")

print("Tuned model:")
calculate_accuracy(hyp_csvs, clean_testref_files[0], "hyperparameters_errors")

Tuned model:
../data/_clean/test/TGL/0156.110816.035620.csv
../data/_clean/testref/TGL/0156.110816.035620.csv
total phones in reference: 9254
total edit distance:       460
phone error rate:          0.04970823427706938


({('q', 'q'): 365,
  ('a', 'a'): 2179,
  ('p', 'p'): 331,
  ('o', 'o'): 397,
  ('s', 's'): 385,
  ('t', 't'): 456,
  ('u', 'u'): 323,
  ('h', 'h'): 157,
  ('d', 'd'): 155,
  ('k', 'k'): 341,
  ('i', 'i'): 759,
  ('n', 'n'): 626,
  ('*', 'q'): 19,
  ('l', 'l'): 392,
  ('y', 'y'): 104,
  ('aw', 'aw'): 13,
  ('ng', 'ng'): 626,
  ('ay', 'ay'): 61,
  ('m', 'm'): 394,
  ('r', 'r'): 125,
  ('e', 'e'): 54,
  ('b', 'b'): 207,
  ('g', 'g'): 273,
  ('u', 'o'): 3,
  ('*', 'a'): 108,
  ('ay', 'y'): 10,
  ('er', 'er'): 12,
  ('sh', 'sh'): 8,
  ('ch', 'ch'): 4,
  ('q', '*'): 20,
  ('w', 'w'): 83,
  ('a', '*'): 10,
  ('i', 'ay'): 2,
  ('o', 'u'): 47,
  ('r', 'er'): 3,
  ('g', 'k'): 1,
  ('*', 'g'): 4,
  ('*', 'n'): 100,
  ('s', '*'): 13,
  ('*', 's'): 4,
  ('*', 'i'): 7,
  ('sh', 'y'): 4,
  ('u', '*'): 4,
  ('l', '*'): 4,
  ('o', '*'): 4,
  ('y', '*'): 1,
  ('ng', '*'): 2,
  ('k', '*'): 1,
  ('er', '*'): 1,
  ('b', '*'): 1,
  ('a', 'l'): 1,
  ('ny', 'ny'): 12,
  ('u', 'a'): 5,
  ('i', '*'): 10,
  ('y'

## 6. Error analysis

This section is not written yet.

In [16]:
Most common

SyntaxError: invalid syntax (314424916.py, line 1)

## 7. Conclusion

The model trained on the cleaned, Filipino-only data outperformed the uncleaned baseline of
Aquino et al. Hyperparameter optimization did not improve it further.

These results suggest that the best WFST G2P model for Filipino is one trained and evaluated on a
Filipino-only dataset with the default Phonetisaurus settings.

## 8. Phoneme inventory

The training corpus of the clean model holds every aligned grapheme-phoneme pair, written as
`grapheme}phoneme`, with `|` joining multi-symbol units and `_` marking an empty side. Collecting
the phoneme side gives the model's phoneme set, saved to `output/phonemes.txt`.

The corpus is a training by-product, so it lives in the temporary directory. Run this after
section 4 and before the clean-up cell.

In [ ]:
corpus = tmp_dir / "train" / "clean_model.corpus"

phonemes = set()

with open(corpus) as f:
    for line in f:
        for unit in line.strip().split():
            if "}" in unit:
                _, phone = unit.split("}", 1)

                for p in phone.split("|"):
                    if p != "_":
                        phonemes.add(p)

phonemes_txt = current_dir / "output" / "phonemes.txt"

with open(phonemes_txt, "w") as f:
    for phone in sorted(phonemes):
        f.write(phone + "\n")

## 9. Hyperparameter optimization

A grid search over six Phonetisaurus settings, scored by 5-fold cross-validation on
`dicts/train.dict`, following the method in Section 4.2.4. The grid (Table 4.1) has **320**
configurations:

| Setting | Values |
| --- | --- |
| `ngram_order` | 3, 4, 5, 6, 7 |
| `seq1_max` | 1, 2, 3, 4 |
| `seq2_max` | 1, 2 |
| `seq1_del` | on, off |
| `seq2_del` | on, off |
| `grow` | on, off |

### 9.1 Split the training dictionary into folds

The entries of `dicts/train.dict` are shuffled with a fixed seed (42), so the folds are the same
on every run, and cut into five folds of about 333 entries each.

In [ ]:
# Load training dictionary
train_dict_path = dicts_dir / "train.dict"
with open(train_dict_path) as f:
    train_entries = [line.strip() for line in f if line.strip()]

print(f"Total training entries: {len(train_entries)}")

# Hyperparameter grid (Table 4.1)
param_grid = {
    "ngram_order": [3, 4, 5, 6, 7],
    "seq1_max":    [1, 2, 3, 4],
    "seq2_max":    [1, 2],
    "seq1_del":    [True, False],
    "seq2_del":    [True, False],
    "grow":        [True, False],
}

keys = list(param_grid.keys())
configs = list(itertools.product(*param_grid.values()))
print(f"Total configurations: {len(configs)}")

# 5-fold word-level split (fixed seed for reproducibility)
random.seed(42)
shuffled = train_entries.copy()
random.shuffle(shuffled)

N_FOLDS = 5
fold_size = len(shuffled) // N_FOLDS
folds = [shuffled[i * fold_size:(i + 1) * fold_size] for i in range(N_FOLDS)]
folds[-1] += shuffled[N_FOLDS * fold_size:]  # remainder into last fold
print(f"Fold sizes: {[len(f) for f in folds]}")

Total training entries: 1667
Total configurations: 320
Fold sizes: [333, 333, 333, 333, 335]


### 9.2 Run the grid search

Each configuration is trained on four folds and tested on the fifth, once per fold. Every fold
runs in its own temporary directory.

- A fold that fails to train counts as PER 1.0.
- A fold that takes longer than 120 seconds also counts as 1.0, and the configuration's
  remaining folds are skipped.

The mean PER of each configuration is saved to `output/hyperparam_results.csv` after every
configuration, so progress isn't lost if the search stops.

In [ ]:
results_path = current_dir / "output" / "hyperparam_results.csv"
results_path.parent.mkdir(parents=True, exist_ok=True)

rows = []

for config_idx, config_vals in enumerate(configs):
    config = dict(zip(keys, config_vals))
    fold_pers = []

    for k in range(N_FOLDS):
        held_out  = folds[k]
        train_fold = [e for i, fold in enumerate(folds) if i != k for e in fold]

        with tempfile.TemporaryDirectory() as tmpdir:
            tmpdir = Path(tmpdir)

            # Write fold training dict
            fold_dict = tmpdir / "fold.dict"
            fold_dict.write_text("\n".join(train_fold))

            # Write held-out word list (deduplicated, preserving order)
            seen = set()
            wlist_lines = []
            for line in held_out:
                parts = line.split()
                if parts and parts[0] not in seen:
                    seen.add(parts[0])
                    wlist_lines.append(parts[0])
            held_wlist = tmpdir / "held.wlist"
            held_wlist.write_text("\n".join(wlist_lines))

            # Build training command
            cmd = [
                "phonetisaurus-train",
                "--lexicon",     str(fold_dict),
                "--ngram_order", str(config["ngram_order"]),
                "--seq1_max",    str(config["seq1_max"]),
                "--seq2_max",    str(config["seq2_max"]),
            ]
            if config["seq1_del"]: cmd.append("--seq1_del")
            if config["seq2_del"]: cmd.append("--seq2_del")
            if config["grow"]:     cmd.append("--grow")
            try:
                train_result = subprocess.run(
                    cmd, cwd=tmpdir, capture_output=True, text=True, timeout=120
                )
            except subprocess.TimeoutExpired:
                print("TIMEOUT:", config)
                fold_pers.append(1.0)
                break

            if train_result.returncode != 0:
                fold_pers.append(1.0)  # penalise failed configs
                continue

            # Apply model to held-out words
            apply_result = subprocess.run(
                [
                    "phonetisaurus-apply",
                    "--model",     str(tmpdir / "train" / "tuned_model.fst"),
                    "--word_list", str(held_wlist),
                ],
                capture_output=True, text=True, cwd=tmpdir
            )

            fold_pers.append(compute_per_cv(held_out, apply_result.stdout))

    mean_per = sum(fold_pers) / len(fold_pers)
    rows.append({**config, "mean_per": mean_per})

    # Save incrementally so progress isn't lost
    pd.DataFrame(rows).to_csv(results_path, index=False)

    if (config_idx + 1) % 10 == 0:
        best_so_far = min(r["mean_per"] for r in rows)
        print(f"[{config_idx + 1}/{len(configs)}] best so far: {best_so_far:.4f}")

print("Grid search complete.")

TIMEOUT: {'ngram_order': 3, 'seq1_max': 1, 'seq2_max': 1, 'seq1_del': True, 'seq2_del': False, 'grow': True}
TIMEOUT: {'ngram_order': 3, 'seq1_max': 1, 'seq2_max': 1, 'seq1_del': False, 'seq2_del': False, 'grow': True}
[10/320] best so far: 1.0000
TIMEOUT: {'ngram_order': 3, 'seq1_max': 1, 'seq2_max': 2, 'seq1_del': True, 'seq2_del': False, 'grow': True}
TIMEOUT: {'ngram_order': 3, 'seq1_max': 1, 'seq2_max': 2, 'seq1_del': False, 'seq2_del': False, 'grow': True}
TIMEOUT: {'ngram_order': 3, 'seq1_max': 2, 'seq2_max': 1, 'seq1_del': True, 'seq2_del': False, 'grow': True}
[20/320] best so far: 1.0000
TIMEOUT: {'ngram_order': 3, 'seq1_max': 2, 'seq2_max': 1, 'seq1_del': False, 'seq2_del': False, 'grow': True}
TIMEOUT: {'ngram_order': 3, 'seq1_max': 2, 'seq2_max': 2, 'seq1_del': True, 'seq2_del': False, 'grow': True}
[30/320] best so far: 1.0000
TIMEOUT: {'ngram_order': 3, 'seq1_max': 2, 'seq2_max': 2, 'seq1_del': False, 'seq2_del': False, 'grow': True}
TIMEOUT: {'ngram_order': 3, 'seq1_max

### 9.3 Pick the best configuration

The configurations are ranked by mean PER. The best one (`ngram_order` 4, `seq1_max` 1,
`seq2_max` 2, with `seq2_del` and `grow` on) reached a mean PER of **7.88%**. Section 5 uses these
settings.

In [ ]:
results_df = pd.read_csv(results_path).sort_values("mean_per").reset_index(drop=True)

print("Top 10 configurations:")
print(results_df.head(10).to_string(index=False))

best = results_df.iloc[0]
print(f"\nBest config:")
for k, v in best.items():
    print(f"  {k}: {v}")

Top 10 configurations:
 ngram_order  seq1_max  seq2_max  seq1_del  seq2_del  grow  mean_per
           4         1         2     False      True  True  0.078817
           3         1         2     False      True  True  0.079057
           5         1         2     False      True  True  0.079346
           4         1         2     False      True False  0.079497
           5         1         2     False      True False  0.080025
           3         1         2     False      True False  0.081234
           5         2         2      True      True False  0.081799
           5         2         2      True      True  True  0.081799
           5         2         2      True     False False  0.081800
           5         2         2     False      True  True  0.081896

Best config:
  ngram_order: 4
  seq1_max: 1
  seq2_max: 2
  seq1_del: False
  seq2_del: True
  grow: True
  mean_per: 0.0788167864523166


### 9.4 Retrain on the full dictionary

The best configuration is retrained on all of `dicts/train.dict`. It is saved as `train/model.fst`,
which replaces the baseline model from section 3.

In [ ]:
best_flags = [
    "--ngram_order", str(int(best["ngram_order"])),
    "--seq1_max",    str(int(best["seq1_max"])),
    "--seq2_max",    str(int(best["seq2_max"])),
]
if best["seq1_del"]: best_flags.append("--seq1_del")
if best["seq2_del"]: best_flags.append("--seq2_del")
if best["grow"]:     best_flags.append("--grow")

print("Training flags:", " ".join(best_flags))

result = train_model(dicts_dir / "train.dict", "model", model_dir, tmp_dir / "train", *best_flags)
print(result.stderr)

Training command: phonetisaurus-train --lexicon /workspaces/Taglog-G2P/notebook/dicts/train.dict --ngram_order 4 --seq1_max 1 --seq2_max 2 --seq2_del --grow
INFO:phonetisaurus-train:2026-05-05 08:07:01:  Checking command configuration...
INFO:phonetisaurus-train:2026-05-05 08:07:01:  Checking lexicon for reserved characters: '}', '|', '_'...
INFO:phonetisaurus-train:2026-05-05 08:07:01:  Aligning lexicon...
INFO:phonetisaurus-train:2026-05-05 08:07:18:  Training joint ngram model...
INFO:phonetisaurus-train:2026-05-05 08:07:20:  Converting ARPA format joint n-gram model to WFST format...
INFO:phonetisaurus-train:2026-05-05 08:07:21:  G2P training succeeded: train/model.fst



### 9.5 Evaluate on the test split

The retrained model is scored on the `data/_small` test split, using the word list in
`wlist/test.wlist`. In the last run it reached a PER of **5.90%**.

In [ ]:
tuned_output_txt = current_dir / "output" / "tuned_output.txt"

with open(tuned_output_txt, "w") as f:
    subprocess.run(
        [
            "phonetisaurus-apply",
            "--model",     str(model_dir / "model.fst"),
            "--word_list", str(wlist_dir / "test.wlist"),
        ],
        stdout=f, text=True
    )

hyp_csvs = create_trxn(test_log, tuned_output_txt, tmp_dir / "hyp")

print("Tuned model:")
calculate_accuracy(hyp_csvs, testref_files[0])

Tuned model:
../data/_small/test/TGL/0156.110816.035620.csv
../data/_small/testref/TGL/0156.110816.035620.csv
total phones in reference: 11089
total edit distance:       654
phone error rate:          0.05897736495626296


## 10. Clean up

Deletes the temporary directory with the file lists, word lists, training by-products and
predicted test transcriptions. It is also deleted automatically when the kernel shuts down.

In [ ]:
tmp.cleanup()